# 07 - Chronological LightGBM Prediction & Evaluation

Performs leak-free chronological splitting (Train/Val/Test), trains LightGBM, and computes MAE, RMSE, and R².


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.data_loader import load_all_raw_data
from src.preprocessing import combine_and_clean_datasets
from src.hampel_filter import run_hampel_pipeline
from src.ceemdan import run_ceemdan_pipeline
from src.reconstruction import run_reconstruction_pipeline
from src.feature_engineering import generate_features
from src.model import chronological_train_val_test_split, train_lightgbm_model
from src.evaluation import run_evaluation_pipeline

_, df_clean = combine_and_clean_datasets(*load_all_raw_data())
df_hampel = run_hampel_pipeline(df_clean)
df_imfs, imfs = run_ceemdan_pipeline(df_hampel)
df_recon, _, _ = run_reconstruction_pipeline(df_imfs, imfs)
df_ready, feature_cols, target = generate_features(df_recon)

splits = chronological_train_val_test_split(df_ready, feature_cols, target)
model = train_lightgbm_model(splits['train'][0], splits['train'][1], splits['val'][0], splits['val'][1])
metrics = run_evaluation_pipeline(model, splits)


In [ ]:
print('Test Metrics:', metrics['test'])
